In [ ]:
import os
print(os.getcwd())

In [ ]:
import json
import glob

def load_listen_data(service):
  out = []
  files = glob.glob(f'../data/{service}/*.json')
  for file in files:
    with open(file, 'r') as f:
      out.extend(json.load(f))
  
  return out
  

spotify_data = load_listen_data('spotify')
lastfm_data = load_listen_data('lastfm')

In [ ]:
import pandas as pd

spotify_df = pd.DataFrame.from_dict(spotify_data)
spotify_df['ts'] = pd.to_datetime(spotify_df['ts'])
spotify_df['source'] = 'spotify'
spotify_df = spotify_df.dropna(subset=['spotify_track_uri']) # these represent podcasts, not music
display(spotify_df)

In [ ]:
def extract_fields(track):
  return {
    'source': 'lastfm',
    'master_metadata_album_artist_name': track['artist']['#text'],
    'master_metadata_album_album_name': track['album']['#text'],
    'master_metadata_track_name': track['name'],
    'ts': pd.to_datetime(int(track['date']['uts']), unit='s', utc=True)
  }

def flatten_lastfm_data(data):
  out = []
  
  for page in data:
    tracks = page['track']
    out.extend(map(extract_fields, tracks))
  
  return out

flat_lasfm_data = flatten_lastfm_data(lastfm_data)

In [ ]:
lastfm_df = pd.DataFrame.from_dict(flat_lasfm_data)
display(lastfm_df)

In [ ]:
listens_df = pd.concat([spotify_df, lastfm_df])
display(listens_df.describe(include='all'))

In [ ]:
import seaborn as sns

listens_df['year'] = listens_df['ts'].dt.year

sns.histplot(listens_df, x='year',  hue='source')

In [ ]:
listens_df.value_counts(subset=['master_metadata_album_artist_name', 'master_metadata_album_album_name', 'master_metadata_track_name'])

## API call definitions

In [ ]:
import musicbrainzngs as mb
import numpy as np

mb.set_rate_limit()
mb.set_useragent('trydionel-ds-project', '0.0.1', 'jeff@trydionel.com')

def search_recordings(track):
  res = mb.search_recordings(limit=1, artist=track['master_metadata_album_artist_name'], release=track['master_metadata_album_album_name'], recording=track['master_metadata_track_name'])
  recordings = res['recording-list']

  if len(recordings) == 0:
    return None

  if int(recordings[0]['ext:score']) < 95:
    return None

  return recordings[0]

def search_albums(track):
  try:
    res = mb.search_releases(limit=1, artist=track['master_metadata_album_artist_name'], release=track['master_metadata_album_album_name'])
    recordings = res['release-list']

    if len(recordings) == 0:
      return None

    if int(recordings[0]['ext:score']) < 95:
      return None

    return recordings[0]
  except mb.NetworkError as e:
    print("Unable to fetch info")
    return {}

def search_artists(track):
  try:
    res = mb.search_artists(limit=1, artist=track['master_metadata_album_artist_name'])
    recordings = res['artist-list']

    if len(recordings) == 0:
      return None

    if int(recordings[0]['ext:score']) < 95:
      return None

    return recordings[0]
  except mb.NetworkError as e:
    print("Unable to fetch info")
    return {}

import requests

def get_audio_features(track):
  if not track['spotify_track_uri']:
    return None

  id = track['spotify_track_uri'].split(':')[-1]

  headers = {
    'Accept': 'application/json'
  }
  res = requests.get("https://api.reccobeats.com/v1/audio-features", headers=headers, params={ "ids": [id] })
  return res.json()

sample = listens_df.iloc[np.random.randint(0, len(listens_df)), :]
display(sample)

print("== Recording ==")
mb_recording_data = search_recordings(sample)
display(mb_recording_data)

print("== Album ==")
mb_album_data = search_albums(sample)
display(mb_album_data)

print("== Artist ==")
mb_artist_data = search_artists(sample)
display(mb_artist_data)

print("== Audio features ==")
audio_features = get_audio_features(sample)
print(json.dumps(audio_features, indent=2))

## Artist analysis

* Genres

In [ ]:
def peak_year(ts):
  plays_by_year = ts.dt.year.value_counts()
  return np.average(plays_by_year.index.values, weights=plays_by_year.values).round()


df_artists = listens_df.groupby(by='master_metadata_album_artist_name').agg(
  total_plays=('ts','count'),
  total_playtime_ms=('ms_played','sum'),
  unique_tracks_played=('master_metadata_track_name', 'nunique'),
  unique_albums_played=('master_metadata_album_album_name', 'nunique'),
  years_listened=('ts', lambda ts: ts.dt.year.nunique()),
  most_recent_year=('ts', lambda ts: ts.dt.year.max()),
  peak_year=('ts', peak_year)
).reset_index()

df_top_artists = df_artists.sort_values(by='total_plays', ascending=False).head(50)

display(df_top_artists)

In [ ]:
def get_artist_genre(row):
  try:
    data = search_artists(row)
    tags = sorted(data['tag-list'], key=lambda d: int(d['count']), reverse=True)
    return [tag['name'] for tag in tags[:3]] # top 3 genres reported
  except KeyError:
    return []
df_top_artists['mb_genres'] = df_top_artists.apply(get_artist_genre, axis=1)
display(df_top_artists)

In [ ]:
def load_following_data():
  out = []
  files = glob.glob(f'../data/followed_artists/*.json')
  for file in files:
    with open(file, 'r') as f:
      data = json.load(f)
      out.extend(data['artists']['items'])
  
  return out

following_data = load_following_data()
df_following = pd.DataFrame.from_dict(following_data)

df_following = df_following.rename(columns={"genres":"spotify_genres"})
df_following = df_following.drop(columns=['external_urls', 'images', 'type', 'href', 'id'])
df_following['followers'] = df_following['followers'].apply(lambda s: s['total'])
df_following['is_following'] = True

display(df_following)

In [ ]:
df_merged = pd.merge(left=df_top_artists, right=df_following, left_on='master_metadata_album_artist_name', right_on='name', how='left')

def merge_genres(df):
  spotify_genres = df['spotify_genres'] if isinstance(df['spotify_genres'], list) else []
  mb_genres = df['mb_genres'] if isinstance(df['mb_genres'], list) else []

  return list(set(spotify_genres) | set(mb_genres))

df_merged['genres'] = df_merged.apply(merge_genres, axis=1)
df_merged = df_merged.drop(columns=['name', 'spotify_genres', 'mb_genres'])
df_merged['is_following'] = df_merged['is_following'].fillna(False)

display(df_merged)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

def explode_genres(df):
  mlb = MultiLabelBinarizer()
  genre_matrix = mlb.fit_transform(df['genres'])
  df_genres = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df.index)
  df_out = pd.concat([df[df.columns.difference(['genres'])], df_genres], axis=1)

  return df_out

display(explode_genres(df_merged))

## Album analysis

Metadata:

* Total tracks on album
* Release date
* Album type
* Record label

In [ ]:
df_albums = listens_df[listens_df['master_metadata_album_album_name'] != ''].groupby(by='master_metadata_album_album_name').agg(
  master_metadata_album_artist_name=('master_metadata_album_artist_name', 'first'),
  total_plays=('ts','count'),
  total_playtime_ms=('ms_played','sum'),
  unique_tracks_played=('master_metadata_track_name', lambda track: track.str.lower().nunique()), # FIXME: use levenshtein distance to clean up track names?
  years_listened=('ts', lambda ts: ts.dt.year.nunique()),
  most_recent_year=('ts', lambda ts: ts.dt.year.max()),
  peak_year=('ts', peak_year)
).reset_index()

df_top_albums = df_albums.sort_values(by='total_plays', ascending=False).head(50)

display(df_top_albums)

In [ ]:
def get_album_metadata(row):
  data = search_albums(row)

  try:
    total_tracks = data['medium-track-count']
  except:
    total_tracks = None

  try:
    release_date = data['release-event-list'][0]['date']
  except:
    release_date = None
  
  try:
    album_type = data['release-group']['type']
  except:
    album_type = None
  
  try:
    record_label = data['label-info-list'][0]['label']['name']
  except:
    record_label = None

  return pd.Series({
    'total_tracks': total_tracks,
    'release_date': release_date,
    'album_type': album_type,
    'record_label': record_label,
  })

df_album_metadata = df_top_albums.apply(get_album_metadata, axis=1)
df_top_albums = pd.concat([df_top_albums, df_album_metadata], axis=1)
display(df_top_albums)

In [ ]:
df_top_albums.dtypes

In [ ]:
df_top_albums['percent_tracks_played'] = df_top_albums.apply(lambda r: min(1, r['unique_tracks_played'] / r["total_tracks"]), axis=1)
display(df_top_albums)

## Track analysis

In [ ]:
df_tracks = listens_df.groupby(by='spotify_track_uri').agg(
  master_metadata_album_artist_name=('master_metadata_album_artist_name', 'first'),
  master_metadata_album_album_name=('master_metadata_album_album_name', 'first'),
  master_metadata_track_name=('master_metadata_track_name', 'first'),
  total_plays=('ts','count'),
  total_playtime_ms=('ms_played','sum'),
  unique_tracks_played=('master_metadata_track_name', lambda track: track.str.lower().nunique()), # FIXME: use levenshtein distance to clean up track names?
  years_listened=('ts', lambda ts: ts.dt.year.nunique()),
  most_recent_year=('ts', lambda ts: ts.dt.year.max()),
  peak_year=('ts', peak_year)
).reset_index()

df_top_tracks = df_tracks.sort_values(by='total_plays', ascending=False).head(50)

display(df_top_tracks)

In [ ]:
import time

def enrich_track(track):
  time.sleep(1) # avoid rate limits
  features = get_audio_features(track)
  try:
    metadata = features['content'][0]
  except:
    metadata = {}

  return pd.Series(metadata)

df_top_tracks_metadata = df_top_tracks.apply(enrich_track, axis=1)
df_top_tracks = pd.concat([df_top_tracks, df_top_tracks_metadata], axis=1)
display(df_top_tracks)

## Data cleaning

* Drop track subtext (parenthesis, dash)

In [ ]:
set(list(listens_df[listens_df['master_metadata_album_album_name'] == 'The Downward Spiral']['master_metadata_track_name']))

In [ ]:
display(set(listens_df[listens_df["master_metadata_track_name"].str.match(".+\\s[\\-\\(].+$")]["master_metadata_track_name"]))

In [ ]:
listens_df["master_metadata_track_name_clean"] = listens_df["master_metadata_track_name"].str.replace("\\s[\\-\\(].+$", "", regex=True).str.lower()

In [ ]:
set(list(listens_df[listens_df['master_metadata_album_album_name'] == 'The Downward Spiral']['master_metadata_track_name_clean']))

## ETL flow

In [ ]:
from prefect import task, flow
from prefect.cache_policies import TASK_SOURCE, INPUTS
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import numpy as np
import musicbrainzngs as mb
import json
import glob

@task
def prep_musicbrainz():
  mb.set_rate_limit()
  mb.set_useragent('trydionel-ds-project', '0.0.1', 'jeff@trydionel.com')

@task()
def load_spotify_data():
    out = []
    files = glob.glob(f'../data/spotify/*.json')
    for file in files:
      with open(file, 'r') as f:
        out.extend(json.load(f))
    
    return out

@task()
def build_df_listens(spotify_data):
  spotify_df = pd.DataFrame.from_dict(spotify_data)
  spotify_df['ts'] = pd.to_datetime(spotify_df['ts'])
  spotify_df['source'] = 'spotify'
  spotify_df = spotify_df.dropna(subset=['spotify_track_uri']) # these represent podcasts, not music

  return spotify_df

@task()
def extract_artists(df_listens):
  def peak_year(ts):
    plays_by_year = ts.dt.year.value_counts()
    return np.average(plays_by_year.index.values, weights=plays_by_year.values).round()

  df_artists = df_listens.groupby(by='master_metadata_album_artist_name').agg(
    total_plays=('ts','count'),
    total_playtime_ms=('ms_played','sum'),
    unique_tracks_played=('master_metadata_track_name', 'nunique'),
    unique_albums_played=('master_metadata_album_album_name', 'nunique'),
    years_listened=('ts', lambda ts: ts.dt.year.nunique()),
    most_recent_year=('ts', lambda ts: ts.dt.year.max()),
    peak_year=('ts', peak_year)
  ).reset_index()

  return df_artists.sort_values(by="total_plays", ascending=False).head(25)

@task(cache_policy=TASK_SOURCE + INPUTS, retries=3, retry_delay_seconds=[2, 5, 15])
def fetch_genres(artist):
  res = mb.search_artists(limit=1, artist=artist)
  if len(res['artist-list']) == 0:
    return []

  recordings = res['artist-list'][0]

  if int(recordings['ext:score']) < 95:
    return []
  
  if 'tag-list' not in recordings:
    return []

  tag_list = recordings['tag-list']
  sorted_tags = sorted(tag_list, key=lambda d: int(d['count']), reverse=True)
  return [tag['name'] for tag in sorted_tags[:3]] # top 3 genres reported

@task
def enrich_with_genres(df_artists):
  df_artist_genres = df_artists.apply(lambda r: fetch_genres(r["master_metadata_album_artist_name"]), axis=1)
  mlb = MultiLabelBinarizer()
  genre_matrix = mlb.fit_transform(df_artist_genres)
  df_genres = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df_artists.index)
  df_out = pd.concat([df_artists[df_artists.columns.difference(['genres'])], df_genres], axis=1)

  return df_out

@task
def to_csv(df, name):
  df.to_csv(name, index=False)

@flow()
def etl():
  prep_musicbrainz()
  spotify_data = load_spotify_data()
  df_listens = build_df_listens(spotify_data)
  df_artists = extract_artists(df_listens)
  df_artists_enriched = enrich_with_genres(df_artists)

  to_csv(df_artists_enriched, "artists.csv")

etl()


10:04:24.952 | INFO    | Flow run 'blue-chimpanzee' - Beginning flow run 'blue-chimpanzee' for flow 'etl'

10:04:24.958 | INFO    | Task run 'prep_musicbrainz-404' - Finished in state Completed()

10:04:25.229 | INFO    | Task run 'load_spotify_data-5f9' - Finished in state Completed()

10:04:33.085 | INFO    | Task run 'build_df_listens-c9f' - Finished in state Completed()

10:04:34.843 | INFO    | Task run 'extract_artists-7b9' - Finished in state Completed()

...

10:04:58.494 | INFO    | Task run 'fetch_genres-efc' - Finished in state Completed()

10:04:59.457 | INFO    | Task run 'fetch_genres-f3d' - Finished in state Completed()

10:04:59.518 | INFO    | Task run 'enrich_with_genres-aec' - Finished in state Completed()

10:04:59.533 | INFO    | Task run 'to_csv-e33' - Finished in state Completed()

10:04:59.631 | INFO    | Flow run 'blue-chimpanzee' - Finished in state Completed()